# Biblical Qwen3.8 27B DPO Training with Unsloth (4-bit QLoRA)

Phase 2 DPO training on top of the SFT LoRA produced by `biblical_qwen3_8_27b_sft_unsloth_4bit.ipynb`.

- Base model: `unsloth/Qwen3.8-27B` (Unsloth loads its 4-bit re-pack, `unsloth/Qwen3.8-27B-unsloth-bnb-4bit`, which is the base the SFT adapter records)
- SFT LoRA input: `output/biblical_qwen3_8_27b_unsloth_4bit/lora_adapters` (retrained 2026-11-10 on the fixed SFT data)
- DPO data: `data/training-data/biblical_persona_v2/biblical_personas_v4_dpo_local_sft.jsonl` (1,673 pairs; rejected answers written by the SFT adapter itself, filtered by a judge)
- DPO LoRA output: `output/biblical_qwen3_8_27b_unsloth_4bit_dpo_v2/lora_adapters` (run 1's folder was renamed `..._dpo_run1_step25/`; it is kept and only read from)

Run this only after the Qwen3.8 27B SFT notebook has finished and saved its LoRA adapters. **Restart the kernel before running**, so settings from an earlier version of the Setup cell do not linger.

**Written against TRL 1.15.0.** The earlier runs of this notebook used TRL 0.24; the DPO trainer was rewritten in between:

1. `max_prompt_length` no longer exists. Length is filtered in the formatting cell and capped by `max_length`.
2. Pairs are passed as message lists, so TRL renders them through Qwen's own chat template. Each answer then ends with `<|im_end|>`, as in SFT training.
3. For a PEFT model TRL clones the loaded adapter as a frozen `ref` adapter. The reference is therefore the **SFT model**. Under TRL 0.24 the reference was the bare base model.
4. Cached reference log-probs are read only when `trainer.precompute_ref_logps` is true; the cache cell sets it.

**Short run with a stop rule:** beta 0.25, learning rate 1e-6, at most 100 steps, 60 pairs per rejection strategy held out and scored every 5 steps. Training stops when the preferred answers have become more than 1% less likely than under the SFT adapter (`MAX_CHOSEN_DROP_PCT`), and the Save cell keeps the last step inside that limit.

**Run 1 (2026-11-11)** used learning rate 5e-6 with a check every 25 steps and a zero-tolerance stop rule. Reward accuracy reached 1.00 by step 15; at the first check (step 25) the preferred answers were 0.8% less likely than under SFT (-4.4 nats) and the rejected ones 7% less likely (-28.9 nats), so the rule stopped it. Its step-25 adapter was saved by override to `output/biblical_qwen3_8_27b_unsloth_4bit_dpo_run1_step25/lora_adapters` for a side-by-side comparison. This version changes three things: a five times lower learning rate, checks every 5 steps, and a 1% allowance in the stop rule. It reuses run 1's reference log-probs.

**Thinking mode:** `ENABLE_THINKING` affects training here, because DPO prompts end with the generation prompt. See the note in the Setup cell.

**Run status.** The v2 settings have **not been run**. The saved cell outputs below are from run 1 and stay until the notebook is run again.


## 1. Environment Preparation

Install Unsloth and updated HuggingFace libraries (same env-prep as the SFT notebook — the `qwen3_5` architecture requires `transformers`/`peft` from git main).

In [1]:
import os

# =========================== HUGGING FACE CACHE ===========================
# Use the default Hub cache: /root/.cache/huggingface/hub in the unsloth-notebook container,
# which is bind-mounted from /home/spark/.cache/huggingface/hub on the host. Do NOT override
# HF_HUB_CACHE: pointing it at the cache root (not hub/) made unsloth build a second
# models--* tree beside hub/ and re-download every base model.

# Install core packages in the running notebook container
!pip install -q -U unsloth trl accelerate datasets bitsandbytes

# Container ships torchao 0.14.0+git (custom aarch64 build). peft requires
# torchao>=0.16.0 OR torchao absent. No aarch64 wheel >=0.16 on PyPI, so
# uninstall — peft's torchao dispatcher then no-ops and falls through to
# the bnb 4-bit dispatcher, which is what we want for QLoRA anyway.
!pip uninstall -y -q torchao

# Qwen3.8 uses the `qwen3_5` architecture, which may be ahead of PyPI releases.
!pip install -q -U git+https://github.com/huggingface/transformers.git

# Keep PEFT compatible with latest Transformers main.
!pip install -q -U git+https://github.com/huggingface/peft.git

# Purge any copies of these already imported earlier in this kernel (e.g. by another
# library) so the import below picks up the packages just pip-installed above.
# This is what removes the need for a kernel restart.
import sys, importlib
for _name in list(sys.modules):
    if _name.split(".")[0] in ("transformers", "trl", "peft", "unsloth", "unsloth_zoo", "huggingface_hub"):
        del sys.modules[_name]
importlib.invalidate_caches()

# Verify installations
import importlib.util
import unsloth
import transformers
import peft
import trl
print(f"Unsloth:      {unsloth.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"PEFT:         {peft.__version__}")
print(f"TRL:          {trl.__version__}")
print(f"torchao installed: {importlib.util.find_spec('torchao') is not None} (should be False)")
print(f"HF hub cache: {os.environ.get('HF_HUB_CACHE', '<default>')}")
print("Environment ready. No kernel restart needed - continue to Cell 2 (Setup).")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
compressed-tensors 0.17.2a20260707 requires torch>=2.10.0, but you have torch 2.10.0a0+b558c986e8.nv25.11 which is incompatible.
gptqmodel 7.0.0 requires numpy==2.2.6; python_version < "3.14", but you have numpy 2.3.5 which is incompatible.
  DEPRECATION: Setting PIP_CONSTRAINT will not affect build constraints in the future, pip 26.2 will enforce this behaviour change. A possible replacement is to specify build constraints using --build-constraint or PIP_BUILD_CONSTRAINT. To disable this warning without any build constraints set --use-feature=build-constraint or PIP_USE_FEATURE="build-constraint".
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.11.4 requires torchao>=0.13

/usr/local/lib/python3.12/dist-packages/transformers/configuration_utils.py:726: FutureWarning: `rope_scaling` is deprecated and will be removed in a future version. Set `rope_parameters` instead, with `rope_type` and `rope_theta` included.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/unsloth_zoo/temporary_patches/gemma.py:279: FutureWarning: `Gemma3ProcessorKwargs` is deprecated and will be removed in version 5.29.0. Use `ProcessingKwargs` instead.
  from transformers.models.gemma3.processing_gemma3 import Gemma3ProcessorKwargs


🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth:      2026.11.4
Transformers: 5.23.0.dev0
PEFT:         0.21.3.dev0
TRL:          1.15.0
torchao installed: False (should be False)
HF hub cache: <default>
Environment ready. No kernel restart needed - continue to Cell 2 (Setup).


## 2. Setup

In [2]:
import os, sys, subprocess, importlib, json
from pathlib import Path

# DGX Spark (sm_120 / GB10): disable Unsloth's flex_attention override and broken
# torch.compile path. Must be set BEFORE `import unsloth`.
os.environ["UNSLOTH_ENABLE_FLEX_ATTENTION"] = "0"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
# No PYTORCH_CUDA_ALLOC_CONF and no set_per_process_memory_fraction, per
# docs/dpo_notebook_guidelines.md ("New DPO notebooks should set neither"). The training
# cell prints allocated/reserved every CLEANUP_STEPS so a climb is visible early.
if "max_split_size_mb" in os.environ.get("PYTORCH_CUDA_ALLOC_CONF", ""):
    raise RuntimeError("PYTORCH_CUDA_ALLOC_CONF with max_split_size_mb is set in this kernel "
                       "(left over from an earlier version of this cell). Restart the kernel.")

# =========================== HUGGING FACE CACHE ===========================
# Use the default Hub cache: /root/.cache/huggingface/hub in the unsloth-notebook container,
# which is bind-mounted from /home/spark/.cache/huggingface/hub on the host. Do NOT override
# HF_HUB_CACHE: pointing it at the cache root (not hub/) made unsloth build a second
# models--* tree beside hub/ and re-download every base model.

def _pip(*args):
    result = subprocess.run([sys.executable, "-m", "pip", *args], capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-1000:] if result.stderr else result.stdout[-1000:])
        raise RuntimeError(f"pip failed: {' '.join(args)}")

def _check_import(module_name):
    try:
        return importlib.import_module(module_name)
    except (ImportError, ModuleNotFoundError):
        return None

import torch
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU available. Run this in the unsloth-notebook container.")

for module, install_args in {
    "psutil": ["install", "-q", "psutil"],
    "matplotlib": ["install", "-q", "matplotlib"],
    "ipywidgets": ["install", "-q", "ipywidgets"],
}.items():
    if _check_import(module) is None:
        print(f"Installing {install_args[-1]}...")
        _pip(*install_args)

import unsloth          # before transformers, so its patches apply
import transformers, peft, trl

if os.path.exists("/workspace/training/biblical"):
    PROJECT_ROOT = Path("/workspace/training/biblical")
elif os.path.exists("/workspace/biblical"):
    PROJECT_ROOT = Path("/workspace/biblical")
else:
    PROJECT_ROOT = Path("/home/spark/projects/training/biblical")

BASE_LLM = "unsloth/Qwen3.8-27B"
# Source SFT notebook: biblical_qwen3_8_27b_sft_unsloth_4bit.ipynb
SFT_MODEL_NAME_BASE = "biblical_qwen3_8_27b_unsloth_4bit"
# v2: run 1 (now ..._dpo_run1_step25, lr 5e-6, check every 25 steps) solved the task by step 15 and was
# stopped at its first check; see "Run 1" at the top. A new name keeps run 1's step-25 adapter
# and stops the Train cell from picking up run 1's checkpoint.
MODEL_NAME_BASE = "biblical_qwen3_8_27b_unsloth_4bit_dpo_v2"
RUN1_MODEL_NAME_BASE = "biblical_qwen3_8_27b_unsloth_4bit_dpo_run1_step25"   # folder renamed after run 1
# Pairs from notebooks/datagen/biblical_datagen_v2_dpo-local-sft.ipynb: the rejected answers
# were written by the SFT adapter itself and filtered by a judge.
DPO_DATA_FILE = PROJECT_ROOT / "data" / "training-data" / "biblical_persona_v2" / "biblical_personas_v4_dpo_local_sft.jsonl"
SFT_LORA_PATH = PROJECT_ROOT / "output" / SFT_MODEL_NAME_BASE / "lora_adapters"
OUTPUT_BASE = PROJECT_ROOT / "output" / MODEL_NAME_BASE
TRAIN_DIR = OUTPUT_BASE / "train"
LORA_OUTPUT_DIR = OUTPUT_BASE / "lora_adapters"
# Run 1's reference log-probs. They depend on the SFT adapter and the formatted pairs, not on
# learning rate, step counts or the output name, so the cache cell reads them from here when
# the fingerprint matches and every row is token-identical. Read-only: nothing under run 1's
# directory is written or deleted.
REF_LOGPROBS_REUSE_DIR = PROJECT_ROOT / "output" / RUN1_MODEL_NAME_BASE / "train" / "ref_logprobs_cache"

# =========================== THINKING MODE ===========================
# Qwen3.8's chat template accepts `enable_thinking`, which changes the generation
# prompt appended by add_generation_prompt=True:
#   False -> '<|im_start|>assistant\n<think>\n\n</think>\n\n'  (empty block; answer directly)
#   True  -> '<|im_start|>assistant\n<think>\n'                 (model opens a reasoning block)
# This flag DOES affect training here: every DPO prompt ends with that generation prompt.
# The chosen/rejected answers in the DPO file are plain answers with no <think> content, and
# the SFT run formatted its data with enable_thinking=False. Leave this False.
# Changing it changes the formatted-dataset fingerprint, so the caches below rebuild.
ENABLE_THINKING = False
# Every completion must end the way SFT trained it. Checked on every row in the formatting cell.
COMPLETION_END_TOKEN = "<|im_end|>"

# =========================== DPO HYPERPARAMETERS ===========================
# MAX_SEQ_LENGTH: measured 2026-11-11 with the SFT adapter's tokenizer and template
# (thinking off) on the 1,673 pairs: prompt + longer answer median 763 tokens, p99 982,
# max 1,247. The formatting cell re-measures and drops (and reports) anything longer.
# TRL 1.x has no max_prompt_length.
MAX_SEQ_LENGTH = 2048
BATCH_SIZE = 1
GRAD_ACCUM = 8
# Run 1 used 5e-6 and reached reward accuracy 1.00 by step 15. Five times lower, so the
# 5-step checks below can see the run before the task is solved.
LEARNING_RATE = 1e-6
# Beta anchors the policy to the SFT reference. The sigmoid loss is saturated at a reward
# margin of about 5, i.e. 5 / beta nats of log-prob shift: 100 nats at 0.05, 20 at 0.25.
DPO_BETA = 0.25
# A hard cap, not one epoch (100 steps x 8 = 800 training pairs seen at most).
MAX_STEPS = 100
WARMUP_STEPS = 10
# Every evaluated step must have a checkpoint, and none may be deleted: the Save cell picks one.
SAVE_STEPS = 5
SAVE_TOTAL_LIMIT = None   # keep all
# Held-out pairs: never trained on, evaluated every EVAL_STEPS (25 in run 1, which was too
# coarse: the first check came after the task was solved).
EVAL_STEPS = 5
# Stop rule. Training stops at the first evaluation where the preferred (chosen) answers have
# become more than this much LESS likely than under the SFT adapter, measured as a percentage
# of their average reference log-prob on the held-out pairs. 0 = stop at any drop at all,
# which with 5-step checks stops on noise (run 1 was already 0.01% down at step 10).
# For scale: run 1 was 0.8% down at step 25.
MAX_CHOSEN_DROP_PCT = 1.0
HOLDOUT_PER_SOURCE = 60   # per rejection strategy: 120 pairs with the two strategies in the file
HOLDOUT_SEED = 3407
# Save cell: None = the last evaluated step still inside MAX_CHOSEN_DROP_PCT.
# Set to a checkpoint step (e.g. 25) to save that checkpoint regardless.
SAVE_STEP_OVERRIDE = None
CLEANUP_STEPS = 50
LOSS_TYPE = "sigmoid"
DPO_MAX_PAIRS = 0   # 0 = use all valid pairs; set e.g. 500/1000 for capped runs

# ---- Less often changed. Kept here so nothing has to be edited further down. ----
LR_SCHEDULER = "cosine"
WEIGHT_DECAY = 0.01
SEED = 3407
LOGGING_STEPS = 5           # training lines printed every N steps
# Reference log-prob cache. Changing either value changes the cache fingerprint, so the
# log-probs would be computed again instead of reused from run 1.
REF_LOGPROBS_SHARD_SIZE = 64
REF_LOGPROBS_BATCH_SIZE = 1
# Quick Evaluation cell (sampling).
GEN_TEMPERATURE = 0.7
GEN_TOP_P = 0.8
GEN_TOP_K = 20
GEN_MAX_NEW_TOKENS = 512    # used when thinking is off; 1536 when it is on

if EVAL_STEPS % SAVE_STEPS:
    raise ValueError(f"EVAL_STEPS ({EVAL_STEPS}) must be a multiple of SAVE_STEPS ({SAVE_STEPS}): "
                     "the Save cell loads the checkpoint of an evaluated step.")

TRAIN_DIR.mkdir(parents=True, exist_ok=True)

for path, label in [(DPO_DATA_FILE, "DPO data"),
                    (SFT_LORA_PATH / "adapter_config.json", "SFT LoRA config"),
                    (SFT_LORA_PATH / "adapter_model.safetensors", "SFT LoRA weights")]:
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")
# The base the SFT adapter records; the Save cell checks the DPO adapter records the same one.
SFT_ADAPTER_BASE = json.load(open(SFT_LORA_PATH / "adapter_config.json")).get("base_model_name_or_path")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"unsloth {unsloth.__version__} | transformers {transformers.__version__} | "
      f"peft {peft.__version__} | trl {trl.__version__}")
print(f"HF hub cache: {os.environ.get('HF_HUB_CACHE', '<default>')}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Base model:   {BASE_LLM} (SFT adapter records {SFT_ADAPTER_BASE})")
print(f"DPO data:     {DPO_DATA_FILE}")
print(f"SFT LoRA in:  {SFT_LORA_PATH}")
print(f"DPO LoRA out: {LORA_OUTPUT_DIR}")
print(f"Thinking mode: {'ON' if ENABLE_THINKING else 'OFF'} (affects DPO prompts)")
print(f"Max length:   {MAX_SEQ_LENGTH} | batch {BATCH_SIZE} x accum {GRAD_ACCUM} | lr {LEARNING_RATE} | beta {DPO_BETA}")
print(f"Steps:        {MAX_STEPS} max, {WARMUP_STEPS} warmup")
print(f"Checkpoints:  every {SAVE_STEPS} steps, keep {'all' if SAVE_TOTAL_LIMIT is None else SAVE_TOTAL_LIMIT}")
print(f"Held-out:     {HOLDOUT_PER_SOURCE} pairs per rejection strategy, evaluated every {EVAL_STEPS} steps; "
      f"stop when the chosen answers are more than {MAX_CHOSEN_DROP_PCT}% less likely than under SFT")
print(f"Ref logprobs: reuse {REF_LOGPROBS_REUSE_DIR} if its fingerprint matches (read-only)")

GPU: NVIDIA GB10
unsloth 2026.11.4 | transformers 5.23.0.dev0 | peft 0.21.3.dev0 | trl 1.15.0
HF hub cache: <default>
Project root: /workspace/training/biblical
Base model:   unsloth/Qwen3.8-27B (SFT adapter records unsloth/Qwen3.8-27B-unsloth-bnb-4bit)
DPO data:     /workspace/training/biblical/data/training-data/biblical_persona_v2/biblical_personas_v4_dpo_local_sft.jsonl
SFT LoRA in:  /workspace/training/biblical/output/biblical_qwen3_8_27b_unsloth_4bit/lora_adapters
DPO LoRA out: /workspace/training/biblical/output/biblical_qwen3_8_27b_unsloth_4bit_dpo_v2/lora_adapters
Thinking mode: OFF (affects DPO prompts)
Max length:   2048 | batch 1 x accum 8 | lr 1e-06 | beta 0.25
Steps:        100 max, 10 warmup
Checkpoints:  every 5 steps, keep all
Held-out:     60 pairs per rejection strategy, evaluated every 5 steps; stop when the chosen answers are more than 1.0% less likely than under SFT
Ref logprobs: reuse /workspace/training/biblical/output/biblical_qwen3_8_27b_unsloth_4bit_dpo_run1_

## 3. Load Model And Format DPO Dataset Cache

### Verifying the inherited tower freeze

This notebook does **not** call `get_peft_model`. It resumes the SFT adapter and continues training it, which means it inherits that adapter's module set exactly.

So the tower freeze cannot be established here — only **verified**. If the SFT run leaked adapters into `unsloth/Qwen3.8-27B`'s vision or audio tower, or its MTP head, this run inherits the leak, spends hours on it, and produces an adapter vLLM will refuse.

The check below runs immediately after the adapter loads, before any expensive work.

In [3]:
import hashlib, json, os, random, shutil
from collections import Counter, defaultdict
from datasets import Dataset as HFDataset, load_from_disk
from peft import PeftModel
from unsloth import FastLanguageModel

# Load the base in 4-bit, then attach the saved SFT LoRA adapter directly via PeftModel.
# This keeps the adapter trainable and bypasses Unsloth's PEFT-adapter auto-reload path.
model, tokenizer = FastLanguageModel.from_pretrained(
    BASE_LLM,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
# Qwen3.8 is Qwen3_5ForConditionalGeneration, so Unsloth can return a Processor. TRL decides
# between its text and vision pipelines by the TYPE of processing_class, so the inner
# tokenizer is what gets passed to DPOTrainer. `processor` is kept so the saved adapter
# directory has the processor files too.
processor = None
if hasattr(tokenizer, "tokenizer"):
    processor = tokenizer
    tokenizer = processor.tokenizer
    print("  (Extracted tokenizer from Processor - text-only DPO mode)")
if not getattr(tokenizer, "chat_template", None):
    if processor is not None and getattr(processor, "chat_template", None):
        tokenizer.chat_template = processor.chat_template
        print("  (Copied chat template from Processor to tokenizer)")
    else:
        raise RuntimeError("Neither the tokenizer nor the Processor carries a chat template.")
# Qwen ships a dedicated pad token (<|endoftext|>); the fallback is defensive.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

model = PeftModel.from_pretrained(model, str(SFT_LORA_PATH), is_trainable=True)
print(f"Loaded {BASE_LLM} + SFT adapter from {SFT_LORA_PATH}")
print(f"  BOS / EOS / pad: {tokenizer.bos_token!r} / {tokenizer.eos_token!r} / {tokenizer.pad_token!r}")

# ============ VERIFY THE ADAPTER SCOPE AND THE TOWER FREEZE ============
# This is a TEXT-ONLY fine-tune on a multimodal base. Two properties must hold, and they
# fail in different ways, so both are checked BEFORE the trainer exists (DPOTrainer adds a
# second, frozen adapter copy named "ref"):
#
#   1. WHERE ADAPTERS ATTACHED. PEFT matches a target_modules LIST by suffix across the
#      whole module tree - it has no concept of "the language model". Any tower reusing
#      the language model's leaf names (q_proj, gate_proj, ...) collects adapters too.
#   2. WHAT IS TRAINABLE. Under QLoRA base weights are frozen 4-bit and only LoRA A/B
#      carry gradients, so the towers are frozen by construction. Assert it anyway.
#
# See docs/multimodal_and_hybrid_base_models.md.
def _zone_of(param_name):
    """Return the non-language sub-model a parameter belongs to, or None."""
    n = param_name.replace("base_model.model.", "", 1)
    if n.startswith("mtp."):
        return "mtp head"
    if "vision_tower" in n or ".visual." in n or n.startswith("visual."):
        return "vision tower"
    if "audio_tower" in n or n.startswith("audio_tower."):
        return "audio tower"
    if "multi_modal_projector" in n or "embed_vision" in n or "embed_audio" in n:
        return "mm projector"
    return None


_adapted = sorted({
    n.split(".lora_A")[0].split(".lora_B")[0].replace("base_model.model.", "")
    for n, _ in model.named_parameters() if ".lora_A." in n or ".lora_B." in n
})
_stray = [n for n in _adapted if _zone_of(n)]
if not _adapted or _stray:
    raise RuntimeError(
        f"LoRA scope is wrong: {len(_adapted)} module(s) adapted, {len(_stray)} of them outside the "
        f"language model (e.g. {_stray[:3]}). This adapter was produced by the SFT notebook and the "
        "scope is INHERITED - it cannot be fixed here. Re-run SFT with finetune_vision_layers=False. "
        "Do not train - vLLM refuses adapters carrying vision-layer LoRA."
    )
_unfrozen, _zone_params = {}, Counter()
for _n, _p in model.named_parameters():
    _z = _zone_of(_n)
    if _z is None:
        continue
    _zone_params[_z] += 1
    if _p.requires_grad:
        _unfrozen.setdefault(_z, []).append(_n)
_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
if _trainable == 0:
    raise RuntimeError("The SFT adapter loaded with no trainable parameters (is_trainable did not apply).")
print(f"  LoRA adapted {len(_adapted)} modules, all inside the language model. Trainable params: {_trainable:,}")
for _z, _c in sorted(_zone_params.items()):
    _bad = len(_unfrozen.get(_z, []))
    print(f"  {_z:<14} {_c:>5} params  {'FROZEN' if not _bad else str(_bad) + ' TRAINABLE - BAD'}")
if _unfrozen:
    raise RuntimeError(
        f"{sum(len(v) for v in _unfrozen.values())} tower/MTP parameter(s) are trainable "
        f"(e.g. {[n for v in _unfrozen.values() for n in v][:3]}). This is a text-only "
        "fine-tune - they must stay frozen."
    )

# ============================ FORMATTED DATASET CACHE ============================
# TRL 1.x conversational format: `prompt` = [system, user] messages, `chosen` / `rejected` =
# [assistant message]. TRL tokenizes prompt (with add_generation_prompt=True) and
# prompt+completion through the chat template, passing each row's `chat_template_kwargs`,
# and takes the completion as everything after their common prefix.
CHAT_TEMPLATE_KWARGS = {"enable_thinking": ENABLE_THINKING}


def _render_ids(messages, **kw):
    return tokenizer.apply_chat_template(messages, tokenize=True, return_dict=True,
                                         **CHAT_TEMPLATE_KWARGS, **kw)["input_ids"]


def _common_prefix(a, b):
    n = 0
    for x, y in zip(a, b):
        if x != y:
            break
        n += 1
    return n


FORMATTED_CACHE_DIR = TRAIN_DIR / "formatted_dpo_dataset_cache"
FORMATTED_DATASET_DIR = FORMATTED_CACHE_DIR / "dataset"
FORMATTED_MANIFEST = FORMATTED_CACHE_DIR / "manifest.json"
FORMATTED_CACHE_DIR.mkdir(parents=True, exist_ok=True)

format_config = {
    "cache_version": 2,
    "format": "trl-conversational",
    "dpo_data_file": str(DPO_DATA_FILE),
    "dpo_data_mtime_ns": DPO_DATA_FILE.stat().st_mtime_ns,
    "max_seq_length": MAX_SEQ_LENGTH,
    "dpo_max_pairs": DPO_MAX_PAIRS,
    "enable_thinking": ENABLE_THINKING,
    "chat_template_sha256": hashlib.sha256(tokenizer.chat_template.encode("utf-8")).hexdigest(),
    "tokenizer_class": type(tokenizer).__name__,
    "tokenizer_vocab_size": len(tokenizer),
}
format_fingerprint = hashlib.sha256(json.dumps(format_config, sort_keys=True).encode("utf-8")).hexdigest()
manifest = json.load(open(FORMATTED_MANIFEST)) if FORMATTED_MANIFEST.exists() else None

if manifest and manifest.get("fingerprint") == format_fingerprint and FORMATTED_DATASET_DIR.exists():
    dpo_dataset = load_from_disk(str(FORMATTED_DATASET_DIR))
    print(f"Loaded formatted dataset cache: {FORMATTED_DATASET_DIR}")
else:
    with open(DPO_DATA_FILE) as f:
        raw_pairs = [json.loads(line) for line in f]
    print(f"Loaded {len(raw_pairs):,} DPO pairs")
    for source, count in Counter(p.get("source", "unknown") for p in raw_pairs).most_common():
        print(f"  {source:<24} {count:>5}")

    _end_id = tokenizer.convert_tokens_to_ids(COMPLETION_END_TOKEN)
    _prompt_tail = "<|im_start|>assistant\n<think>\n" if ENABLE_THINKING else "<|im_start|>assistant\n<think>\n\n</think>\n\n"
    formatted_pairs, errors, framing_errors = [], [], []
    skipped_too_long, max_total, max_prompt = 0, 0, 0
    for index, pair in enumerate(raw_pairs):
        chosen_msgs = pair.get("chosen", [])
        rejected_msgs = pair.get("rejected", [])
        if len(chosen_msgs) != 3 or len(rejected_msgs) != 3:
            errors.append(f"pair {index}: expected three chosen/rejected messages")
            continue
        if chosen_msgs[:-1] != rejected_msgs[:-1]:
            errors.append(f"pair {index}: prompt messages differ")
            continue
        if [m["role"] for m in chosen_msgs] != ["system", "user", "assistant"] or rejected_msgs[-1]["role"] != "assistant":
            errors.append(f"pair {index}: roles are not system/user/assistant")
            continue
        if not chosen_msgs[-1]["content"].strip() or not rejected_msgs[-1]["content"].strip():
            errors.append(f"pair {index}: empty chosen or rejected answer")
            continue
        if chosen_msgs[-1]["content"] == rejected_msgs[-1]["content"]:
            errors.append(f"pair {index}: chosen and rejected answers are identical")
            continue
        prompt_msgs = chosen_msgs[:-1]
        # Exactly what DPOTrainer does when it tokenizes, so the checks see what training sees.
        prompt_ids = _render_ids(prompt_msgs, add_generation_prompt=True)
        chosen_ids = _render_ids(chosen_msgs)
        rejected_ids = _render_ids(rejected_msgs)
        prompt_len = min(_common_prefix(prompt_ids, chosen_ids), _common_prefix(prompt_ids, rejected_ids))
        c_ids, r_ids = chosen_ids[prompt_len:], rejected_ids[prompt_len:]
        # The prompt must be a true prefix (so the completion is the answer and nothing else),
        # it must end with the generation prompt SFT trained under, and each completion must
        # close with <|im_end|> (the template may add one newline token after it).
        if (prompt_len != len(prompt_ids)
                or not tokenizer.decode(prompt_ids[-12:]).endswith(_prompt_tail)
                or _end_id not in c_ids[-2:] or _end_id not in r_ids[-2:]
                or c_ids.count(_end_id) != 1 or r_ids.count(_end_id) != 1):
            framing_errors.append(index)
            continue
        total = prompt_len + max(len(c_ids), len(r_ids))
        if total > MAX_SEQ_LENGTH:
            skipped_too_long += 1
            continue
        max_total, max_prompt = max(max_total, total), max(max_prompt, prompt_len)
        formatted_pairs.append({
            "prompt": prompt_msgs,
            "chosen": [chosen_msgs[-1]],
            "rejected": [rejected_msgs[-1]],
            "chat_template_kwargs": dict(CHAT_TEMPLATE_KWARGS),
            "source": pair.get("source", "unknown"),
            "persona": pair.get("persona", "unknown"),
        })

    if framing_errors:
        raise RuntimeError(
            f"{len(framing_errors)} pair(s) (first: {framing_errors[:5]}) do not render as "
            f"{_prompt_tail!r} + answer + {COMPLETION_END_TOKEN!r}. The chat template or tokenizer is not "
            "the one this notebook was verified against; do not train on a framing SFT never saw."
        )
    if errors:
        print(f"Validation errors: {len(errors)}")
        for error in errors[:10]:
            print(" ", error)
    if not formatted_pairs:
        raise RuntimeError("No DPO pairs remain after validation/filtering.")

    if DPO_MAX_PAIRS and len(formatted_pairs) > DPO_MAX_PAIRS:
        random.seed(3407)
        by_source = defaultdict(list)
        for pair in formatted_pairs:
            by_source[pair["source"]].append(pair)
        sampled, total_pairs = [], len(formatted_pairs)
        for items in by_source.values():
            sample_count = max(1, round(DPO_MAX_PAIRS * len(items) / total_pairs))
            sampled.extend(random.sample(items, min(sample_count, len(items))))
        random.shuffle(sampled)
        formatted_pairs = sampled[:DPO_MAX_PAIRS]
        print(f"Capped formatted DPO dataset to {len(formatted_pairs):,} pairs")

    dpo_dataset = HFDataset.from_list(formatted_pairs).shuffle(seed=3407)
    if FORMATTED_DATASET_DIR.exists():
        shutil.rmtree(FORMATTED_DATASET_DIR)
    dpo_dataset.save_to_disk(str(FORMATTED_DATASET_DIR))
    tmp_manifest = FORMATTED_MANIFEST.with_suffix(".json.tmp")
    with open(tmp_manifest, "w") as f:
        json.dump({"status": "complete", "fingerprint": format_fingerprint, "config": format_config,
                   "rows": len(dpo_dataset), "max_total_tokens": max_total, "max_prompt_tokens": max_prompt}, f, indent=2)
    os.replace(tmp_manifest, FORMATTED_MANIFEST)
    print(f"Saved formatted dataset cache: {FORMATTED_DATASET_DIR}")
    print(f"  Longest prompt: {max_prompt} tokens | longest prompt+completion: {max_total} tokens (limit {MAX_SEQ_LENGTH})")
    print(f"  Dropped for length: {skipped_too_long:,} | structural errors: {len(errors):,}")
    print(f"  Framing: every prompt ends {_prompt_tail!r}; every completion ends {COMPLETION_END_TOKEN!r}")

print(f"DPO rows: {len(dpo_dataset):,}")

/usr/local/lib/python3.12/dist-packages/unsloth_zoo/temporary_patches/gemma.py:279: FutureWarning: `Gemma3ProcessorKwargs` is deprecated and will be removed in version 5.29.0. Use `ProcessingKwargs` instead.
  from transformers.models.gemma3.processing_gemma3 import Gemma3ProcessorKwargs


==((====))==  Unsloth 2026.11.4: Fast Qwen3_5 patching. Transformers: 5.23.0.dev0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.688 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260302. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/1184 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


  (Extracted tokenizer from Processor - text-only DPO mode)
Loaded unsloth/Qwen3.8-27B + SFT adapter from /workspace/training/biblical/output/biblical_qwen3_8_27b_unsloth_4bit/lora_adapters
  BOS / EOS / pad: None / '<|im_end|>' / '<|endoftext|>'
  LoRA adapted 256 modules, all inside the language model. Trainable params: 159,383,552
  vision tower     333 params  FROZEN
Loaded 1,673 DPO pairs
  voice_drift               1142
  shallow_platitude          531


Saving the dataset (0/1 shards):   0%|          | 0/1673 [00:00<?, ? examples/s]

Saved formatted dataset cache: /workspace/training/biblical/output/biblical_qwen3_8_27b_unsloth_4bit_dpo_v2/train/formatted_dpo_dataset_cache/dataset
  Longest prompt: 432 tokens | longest prompt+completion: 1247 tokens (limit 2048)
  Dropped for length: 0 | structural errors: 0
  Framing: every prompt ends '<|im_start|>assistant\n<think>\n\n</think>\n\n'; every completion ends '<|im_end|>'
DPO rows: 1,673


## 4. Configure DPO Trainer

In [4]:
import inspect, math, torch
from trl import DPOTrainer, DPOConfig

# This notebook is written against the TRL 1.x DPO trainer. Fail here, with a reason,
# rather than hours later inside the cache or the loss.
_ref_sig = list(inspect.signature(DPOTrainer.compute_ref_log_probs).parameters)
if _ref_sig[:3] != ["self", "model", "inputs"]:
    raise RuntimeError(
        f"TRL {trl.__version__}: DPOTrainer.compute_ref_log_probs has signature {_ref_sig}, not "
        "(self, model, inputs). This notebook targets TRL 1.x (verified on 1.15.0)."
    )

FastLanguageModel.for_training(model)
effective_batch = BATCH_SIZE * GRAD_ACCUM
# A hard step cap, not one epoch.
max_steps = MAX_STEPS
warmup_steps = WARMUP_STEPS

trainer = DPOTrainer(
    model=model,
    ref_model=None,            # PEFT: TRL clones the loaded SFT adapter as a frozen "ref" adapter
    args=DPOConfig(
        beta=DPO_BETA,
        loss_type=LOSS_TYPE,
        max_length=MAX_SEQ_LENGTH,
        precompute_ref_log_probs=False,   # the persistent cache cell below handles this
        per_device_train_batch_size=BATCH_SIZE,
        # Held-out evaluation (Train cell). The Trainer default is 8; each forward carries
        # chosen + rejected.
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type=LR_SCHEDULER,
        warmup_steps=warmup_steps,
        max_steps=max_steps,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim="adamw_8bit",
        weight_decay=WEIGHT_DECAY,
        seed=SEED,
        gradient_checkpointing=True,
        dataloader_pin_memory=False,
        output_dir=str(TRAIN_DIR),
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=SAVE_TOTAL_LIMIT,
        logging_steps=LOGGING_STEPS,
        report_to="none",
        dataset_num_proc=1,
    ),
    # All pairs go in here so the reference-logprob cache covers every row. The cache cell
    # then moves the held-out slice to trainer.eval_dataset.
    train_dataset=dpo_dataset,
    processing_class=tokenizer,    # a tokenizer, not the Processor: selects TRL's text pipeline
)

# ---- Assert the trainer took the path this notebook assumes -------------------------
if getattr(trainer, "_is_vlm", False) or getattr(trainer, "_is_vision_dataset", False):
    raise RuntimeError("DPOTrainer chose its vision pipeline. processing_class must be the inner tokenizer.")
_needed = {"prompt_ids", "chosen_ids", "rejected_ids"}
if not _needed.issubset(trainer.train_dataset.column_names):
    raise RuntimeError(f"Tokenized dataset lacks {_needed - set(trainer.train_dataset.column_names)}.")
if len(trainer.train_dataset) != len(dpo_dataset):
    raise RuntimeError(f"TRL dropped rows while tokenizing: {len(dpo_dataset)} -> {len(trainer.train_dataset)}.")
if "source" not in trainer.train_dataset.column_names:
    raise RuntimeError("TRL removed the 'source' column; the held-out split in the next cell needs it.")

# What TRL actually tokenized, checked on real rows: prompt tail, completion ending, length.
_end_id = tokenizer.convert_tokens_to_ids(COMPLETION_END_TOKEN)
_prompt_tail = "<|im_start|>assistant\n<think>\n" if ENABLE_THINKING else "<|im_start|>assistant\n<think>\n\n</think>\n\n"
_bad, _longest = 0, 0
for _row in trainer.train_dataset.select(range(min(200, len(trainer.train_dataset)))):
    if not tokenizer.decode(_row["prompt_ids"][-12:]).endswith(_prompt_tail):
        _bad += 1
    for _k in ("chosen_ids", "rejected_ids"):
        if _end_id not in _row[_k][-2:] or _row[_k].count(_end_id) != 1:
            _bad += 1
    _longest = max(_longest, len(_row["prompt_ids"]) + max(len(_row["chosen_ids"]), len(_row["rejected_ids"])))
if _bad:
    raise RuntimeError(f"{_bad} tokenized prompt(s)/completion(s) in the first 200 rows are not "
                       f"{_prompt_tail!r} + answer + {COMPLETION_END_TOKEN!r}. Do not train.")
_r0 = trainer.train_dataset[0]

# The reference is the frozen "ref" copy of the SFT adapter; only "default" may train.
_adapters = list(model.peft_config)
_ref_trainable = [n for n, p in model.named_parameters() if ".ref." in n and p.requires_grad]
_default_trainable = sum(1 for n, p in model.named_parameters() if ".default." in n and p.requires_grad)
if "ref" not in _adapters:
    raise RuntimeError(f"DPOTrainer did not create the 'ref' adapter (adapters: {_adapters}). With no copy of "
                       "the SFT adapter the reference would be the bare base model, which is not the intent here.")
if _ref_trainable or not _default_trainable:
    raise RuntimeError(f"Adapter trainability is wrong: {len(_ref_trainable)} 'ref' tensors trainable, "
                       f"{_default_trainable} 'default' tensors trainable.")

print("DPO trainer configured.")
print(f"  TRL {trl.__version__} | text pipeline | adapters: {_adapters} (active: {model.active_adapters})")
print(f"  DPO pairs: {len(trainer.train_dataset):,} (before the held-out split) | effective batch {effective_batch} | max steps {max_steps} | warmup {warmup_steps} | beta {trainer.beta}")
print(f"  max_length: {trainer.args.max_length} | longest row in first 200: {_longest} tokens")
print(f"  prompt tail:     {tokenizer.decode(_r0['prompt_ids'][-8:])!r}")
print(f"  completion head: {tokenizer.decode(_r0['chosen_ids'][:6])!r} ... {tokenizer.decode(_r0['chosen_ids'][-2:])!r}")
print("  precompute_ref_log_probs: False at construction; the next cell builds the persistent cache")

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1541: FutureWarning: `gradient_checkpointing` is deprecated and will be removed in v6. Use `activation_checkpointing` instead.
  warnings.warn(f"`{old}` is deprecated and will be removed in v6. Use `{new}` instead.", FutureWarning)


Tokenizing train dataset:   0%|          | 0/1673 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/1673 [00:00<?, ? examples/s]

DPO trainer configured.
  TRL 1.15.0 | text pipeline | adapters: ['default', 'ref'] (active: ['default'])
  DPO pairs: 1,673 (before the held-out split) | effective batch 8 | max steps 100 | warmup 10 | beta 0.25
  max_length: 2048 | longest row in first 200: 951 tokens
  prompt tail:     '\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
  completion head: 'As the fig tree yields her' ... '<|im_end|>\n'
  precompute_ref_log_probs: False at construction; the next cell builds the persistent cache


## 5. Precompute Reference Log Probabilities (Persistent, Resumable Cache)

DPO needs the frozen reference model's log probability of every chosen and rejected completion. TRL's own `precompute_ref_log_probs=True` is one-shot and in-memory, so a kernel crash during it starts over from row 0.

This cell computes them in shards under `TRAIN_DIR/ref_logprobs_cache/shards/`, writes a fingerprinted manifest, resumes from existing shards, attaches `ref_chosen_logps` / `ref_rejected_logps` to the trainer dataset, and then sets `trainer.precompute_ref_logps = True`. That last step matters on TRL 1.x: without it the loss ignores the cached columns and re-runs the reference forward pass on every step.

The reference is the frozen `ref` copy of the SFT adapter that `DPOTrainer` created, so the fingerprint includes the SFT adapter file: a retrained SFT adapter rebuilds the cache.

**Reuse.** The fingerprint does not depend on learning rate, step counts or the output name, so run 1's finished cache (`REF_LOGPROBS_REUSE_DIR`) is loaded as it is when its fingerprint matches and every row is token-identical. It is only read; if it does not match, a new cache is built under this run's own `TRAIN_DIR`.

**Held-out split.** After the log-probs are attached, the cell moves `HOLDOUT_PER_SOURCE` pairs per rejection strategy to `trainer.eval_dataset` and drops any training pair that shares its prompt and chosen answer with a held-out pair. The printed line gives the exact counts.

In [5]:
import hashlib, json, os, random, time, shutil
from collections import Counter, defaultdict
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from datasets import load_from_disk

# Re-running this cell: start again from the full tokenized set, not from the already-split one.
if getattr(trainer, "_dpo_full_dataset", None) is not None:
    trainer.train_dataset = trainer._dpo_full_dataset

REF_LOGPROBS_CACHE_DIR = TRAIN_DIR / "ref_logprobs_cache"
REF_LOGPROBS_SHARD_DIR = REF_LOGPROBS_CACHE_DIR / "shards"
REF_LOGPROBS_DATASET_DIR = REF_LOGPROBS_CACHE_DIR / "dataset"
REF_LOGPROBS_MANIFEST = REF_LOGPROBS_CACHE_DIR / "manifest.json"
# REF_LOGPROBS_SHARD_SIZE and REF_LOGPROBS_BATCH_SIZE are set in the Setup cell.

REF_LOGPROBS_CACHE_DIR.mkdir(parents=True, exist_ok=True)
REF_LOGPROBS_SHARD_DIR.mkdir(parents=True, exist_ok=True)

_sft_weights = SFT_LORA_PATH / "adapter_model.safetensors"
_ref_cache_config = {
    "cache_version": 2,
    "trl_version": trl.__version__,
    "base_llm": BASE_LLM,
    "sft_lora_path": str(SFT_LORA_PATH),
    "sft_adapter_size": _sft_weights.stat().st_size,
    "sft_adapter_mtime_ns": _sft_weights.stat().st_mtime_ns,
    "reference": "ref adapter (copy of SFT)" if "ref" in model.peft_config else "base model",
    "formatted_dataset_fingerprint": format_fingerprint,
    "dataset_len": len(trainer.train_dataset),
    "max_seq_length": MAX_SEQ_LENGTH,
    "enable_thinking": ENABLE_THINKING,
    "shard_size": REF_LOGPROBS_SHARD_SIZE,
    "batch_size": REF_LOGPROBS_BATCH_SIZE,
}
_ref_cache_fingerprint = hashlib.sha256(
    json.dumps(_ref_cache_config, sort_keys=True).encode("utf-8")
).hexdigest()
_required_ref_cols = {"ref_chosen_logps", "ref_rejected_logps"}


def _read_ref_manifest():
    if not REF_LOGPROBS_MANIFEST.exists():
        return None
    with open(REF_LOGPROBS_MANIFEST) as f:
        return json.load(f)


def _load_valid_ref_shard(shard_path, shard_idx, start, end):
    try:
        shard_payload = torch.load(shard_path, map_location="cpu")
    except Exception as exc:
        raise RuntimeError(f"Unreadable ref logprob shard: {shard_path}") from exc
    if not isinstance(shard_payload, dict):
        raise RuntimeError(f"Invalid ref logprob shard payload: {shard_path}")
    if shard_payload.get("fingerprint") != _ref_cache_fingerprint:
        raise RuntimeError(f"Stale ref logprob shard fingerprint: {shard_path}")
    if shard_payload.get("shard_idx") != shard_idx or shard_payload.get("start") != start or shard_payload.get("end") != end:
        raise RuntimeError(f"Ref logprob shard index/range mismatch: {shard_path}")
    for column in ("ref_chosen_logps", "ref_rejected_logps"):
        values = shard_payload.get(column)
        if not isinstance(values, torch.Tensor) or values.numel() != end - start or not torch.isfinite(values).all():
            raise RuntimeError(f"Invalid {column} values in ref logprob shard: {shard_path}")
    return shard_payload


def _write_ref_manifest(status, completed_shards):
    payload = {
        "status": status,
        "fingerprint": _ref_cache_fingerprint,
        "config": _ref_cache_config,
        "completed_shards": completed_shards,
        "updated_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }
    tmp_path = REF_LOGPROBS_MANIFEST.with_suffix(".json.tmp")
    with open(tmp_path, "w") as f:
        json.dump(payload, f, indent=2)
    os.replace(tmp_path, REF_LOGPROBS_MANIFEST)


# ---- Run 1's cache, read-only -------------------------------------------------------
# Same SFT adapter, same formatted pairs, same TRL: the fingerprint is identical, so the
# reference log-probs are too. If anything differs this is skipped and a new cache is built
# under THIS run's TRAIN_DIR. Nothing below writes to or deletes from REF_LOGPROBS_REUSE_DIR.
_ref_cache_source = getattr(trainer, "_dpo_ref_cache_source", str(REF_LOGPROBS_CACHE_DIR))
_reuse_manifest_path = REF_LOGPROBS_REUSE_DIR / "manifest.json"
_reuse_dataset_dir = REF_LOGPROBS_REUSE_DIR / "dataset"
if (not _required_ref_cols.issubset(trainer.train_dataset.column_names)
        and REF_LOGPROBS_REUSE_DIR.resolve() != REF_LOGPROBS_CACHE_DIR.resolve()):
    _reuse_manifest = json.load(open(_reuse_manifest_path)) if _reuse_manifest_path.exists() else None
    if (_reuse_manifest is not None and _reuse_manifest.get("fingerprint") == _ref_cache_fingerprint
            and _reuse_manifest.get("status") == "complete" and _reuse_dataset_dir.exists()):
        _reuse_dataset = load_from_disk(str(_reuse_dataset_dir))
        _same_rows = len(_reuse_dataset) == len(trainer.train_dataset) and _required_ref_cols.issubset(_reuse_dataset.column_names)
        if _same_rows:
            # Row order must be identical or every log-prob would be attached to the wrong pair.
            _id_cols = ["prompt_ids", "chosen_ids", "rejected_ids"]
            _same_rows = all(_ra == _rb for _ra, _rb in zip(trainer.train_dataset.select_columns(_id_cols),
                                                            _reuse_dataset.select_columns(_id_cols)))
        if _same_rows:
            trainer.train_dataset = _reuse_dataset
            _ref_cache_source = trainer._dpo_ref_cache_source = str(REF_LOGPROBS_REUSE_DIR)
            print(f"Loaded run 1's ref logprob cache (read-only, all {len(_reuse_dataset):,} rows token-identical): {_reuse_dataset_dir}")
        else:
            print("Run 1's ref logprob cache has the right fingerprint but different rows; not using it.")
    else:
        print(f"Run 1's ref logprob cache is missing or has a different fingerprint; building a new one under {REF_LOGPROBS_CACHE_DIR}.")

_manifest = _read_ref_manifest()
_cache_matches = _manifest is not None and _manifest.get("fingerprint") == _ref_cache_fingerprint

if (not _required_ref_cols.issubset(trainer.train_dataset.column_names)
        and _cache_matches and _manifest.get("status") == "complete" and REF_LOGPROBS_DATASET_DIR.exists()):
    cached_dataset = load_from_disk(str(REF_LOGPROBS_DATASET_DIR))
    _same_rows = len(cached_dataset) == len(trainer.train_dataset) and _required_ref_cols.issubset(cached_dataset.column_names)
    if _same_rows:
        # Row order must be identical or every log-prob would be attached to the wrong pair.
        _id_cols = ["prompt_ids", "chosen_ids", "rejected_ids"]
        _same_rows = all(_ra == _rb for _ra, _rb in zip(trainer.train_dataset.select_columns(_id_cols),
                                                        cached_dataset.select_columns(_id_cols)))
    if _same_rows:
        trainer.train_dataset = cached_dataset
        print(f"Loaded persistent ref logprob cache (all {len(cached_dataset):,} rows token-identical): {REF_LOGPROBS_DATASET_DIR}")
    else:
        print("Ignoring stale ref logprob dataset cache: rows do not match")
        _cache_matches = False

if not _required_ref_cols.issubset(trainer.train_dataset.column_names):
    if not _cache_matches:
        for stale_shard in REF_LOGPROBS_SHARD_DIR.glob("shard-*.pt"):
            stale_shard.unlink()
        _write_ref_manifest("in_progress", [])

    num_rows = len(trainer.train_dataset)
    shard_ranges = [
        (start, min(start + REF_LOGPROBS_SHARD_SIZE, num_rows))
        for start in range(0, num_rows, REF_LOGPROBS_SHARD_SIZE)
    ]
    print("Persistent reference logprob cache")
    print(f"  Rows: {num_rows} | shards: {len(shard_ranges)} x {REF_LOGPROBS_SHARD_SIZE} | dir: {REF_LOGPROBS_CACHE_DIR}")
    print(f"  Reference: {_ref_cache_config['reference']}")

    # Only the columns the collator reads; the message-list columns stay out of the loader.
    _ref_source = trainer.train_dataset.select_columns(["prompt_ids", "chosen_ids", "rejected_ids"])
    completed = []
    for shard_idx, (start, end) in enumerate(shard_ranges):
        shard_path = REF_LOGPROBS_SHARD_DIR / f"shard-{shard_idx:06d}.pt"
        if shard_path.exists():
            try:
                _load_valid_ref_shard(shard_path, shard_idx, start, end)
            except RuntimeError as exc:
                print(f"Recomputing ref logprob shard {shard_idx}: {exc}")
                shard_path.unlink()
            else:
                completed.append(shard_idx)
                continue

        shard_loader = DataLoader(
            _ref_source.select(range(start, end)),
            batch_size=REF_LOGPROBS_BATCH_SIZE,
            collate_fn=trainer.data_collator,
            num_workers=0,
            pin_memory=False,
            shuffle=False,
        )
        shard_loader = trainer.accelerator.prepare(shard_loader)

        ref_chosen_logps, ref_rejected_logps = [], []
        for padded_batch in tqdm(shard_loader, desc=f"Ref logprobs shard {shard_idx + 1}/{len(shard_ranges)}"):
            # TRL 1.x signature: (model, inputs). With a PEFT model and no ref_model it
            # switches to the frozen "ref" adapter for this forward pass.
            ref_chosen_logp, ref_rejected_logp = trainer.compute_ref_log_probs(trainer.model, padded_batch)
            ref_chosen_logps.append(ref_chosen_logp.detach().float().cpu())
            ref_rejected_logps.append(ref_rejected_logp.detach().float().cpu())
        torch.cuda.empty_cache()

        shard_payload = {
            "fingerprint": _ref_cache_fingerprint,
            "shard_idx": shard_idx,
            "start": start,
            "end": end,
            "ref_chosen_logps": torch.cat(ref_chosen_logps),
            "ref_rejected_logps": torch.cat(ref_rejected_logps),
        }
        if not (torch.isfinite(shard_payload["ref_chosen_logps"]).all() and torch.isfinite(shard_payload["ref_rejected_logps"]).all()):
            raise RuntimeError(f"Non-finite reference log-probs in shard {shard_idx}. Stop and inspect before training.")
        tmp_shard_path = shard_path.with_suffix(".pt.tmp")
        torch.save(shard_payload, tmp_shard_path)
        os.replace(tmp_shard_path, shard_path)
        completed.append(shard_idx)
        _write_ref_manifest("in_progress", completed)

    all_ref_chosen_logps, all_ref_rejected_logps = [], []
    for shard_idx, (start, end) in enumerate(shard_ranges):
        shard_payload = _load_valid_ref_shard(REF_LOGPROBS_SHARD_DIR / f"shard-{shard_idx:06d}.pt", shard_idx, start, end)
        all_ref_chosen_logps.append(shard_payload["ref_chosen_logps"])
        all_ref_rejected_logps.append(shard_payload["ref_rejected_logps"])
    ref_chosen_values = torch.cat(all_ref_chosen_logps).tolist()
    ref_rejected_values = torch.cat(all_ref_rejected_logps).tolist()
    if len(ref_chosen_values) != num_rows or len(ref_rejected_values) != num_rows:
        raise RuntimeError("Ref logprob cache length does not match training dataset")

    train_dataset_with_ref = trainer.train_dataset
    for ref_col in ("ref_chosen_logps", "ref_rejected_logps"):
        if ref_col in train_dataset_with_ref.column_names:
            train_dataset_with_ref = train_dataset_with_ref.remove_columns(ref_col)
    train_dataset_with_ref = train_dataset_with_ref.add_column("ref_chosen_logps", ref_chosen_values)
    train_dataset_with_ref = train_dataset_with_ref.add_column("ref_rejected_logps", ref_rejected_values)

    if REF_LOGPROBS_DATASET_DIR.exists():
        shutil.rmtree(REF_LOGPROBS_DATASET_DIR)
    train_dataset_with_ref.save_to_disk(str(REF_LOGPROBS_DATASET_DIR))
    trainer.train_dataset = train_dataset_with_ref
    _write_ref_manifest("complete", list(range(len(shard_ranges))))
    print(f"Saved persistent ref logprob dataset cache: {REF_LOGPROBS_DATASET_DIR}")

# TRL 1.x reads the cached columns ONLY when this flag is set (DPOTrainer._compute_loss).
# Without it, every training step would re-run the reference forward pass.
trainer.precompute_ref_logps = True

_rc = torch.tensor(trainer.train_dataset["ref_chosen_logps"])
_rr = torch.tensor(trainer.train_dataset["ref_rejected_logps"])
print(f"Ref logprob columns ready: {sorted(_required_ref_cols)}; trainer.precompute_ref_logps = True")
print(f"  ref chosen   mean {_rc.mean():.1f} (min {_rc.min():.1f}, max {_rc.max():.1f})")
print(f"  ref rejected mean {_rr.mean():.1f} (min {_rr.min():.1f}, max {_rr.max():.1f})")
print(f"  SFT reference already prefers chosen on {(_rc > _rr).float().mean() * 100:.1f}% of pairs "
      "(raw summed log-probs; longer answers score lower, so this is a sanity figure, not accuracy)")

# ============================ HELD-OUT SLICE ============================
# Split AFTER the reference log-probs are attached, so the cache above always covers all rows.
# Stratified by rejection strategy, fixed seed. The same question + gold answer can appear
# under more than one strategy (the datagen samples each strategy independently), so any
# training row that shares its prompt and chosen answer with a held-out row is dropped too.
# Otherwise the held-out chosen reward would be measured on answers the run trains on.
_full = trainer.train_dataset
trainer._dpo_full_dataset = _full
_rng = random.Random(HOLDOUT_SEED)
_by_source = defaultdict(list)
for _i, _s in enumerate(_full["source"]):
    _by_source[_s].append(_i)
_holdout_idx = sorted(_i for _s in sorted(_by_source)
                      for _i in _rng.sample(_by_source[_s], min(HOLDOUT_PER_SOURCE, len(_by_source[_s]))))
_keys = [hashlib.sha256(json.dumps([_r["prompt_ids"], _r["chosen_ids"]]).encode("utf-8")).hexdigest()
         for _r in _full.select_columns(["prompt_ids", "chosen_ids"])]
_holdout_set = set(_holdout_idx)
_holdout_keys = {_keys[_i] for _i in _holdout_idx}
_train_idx = [_i for _i in range(len(_full)) if _i not in _holdout_set and _keys[_i] not in _holdout_keys]
HOLDOUT_LEAK_DROPPED = len(_full) - len(_holdout_idx) - len(_train_idx)
if not _holdout_idx or not _train_idx:
    raise RuntimeError("The held-out split left an empty training or held-out set.")

trainer.eval_dataset = _full.select(_holdout_idx)
trainer.train_dataset = _full.select(_train_idx)
if _holdout_keys & {_keys[_i] for _i in _train_idx}:
    raise RuntimeError("A held-out prompt + chosen answer is still present in the training set.")

_he = trainer.eval_dataset
_hc, _hr = torch.tensor(_he["ref_chosen_logps"]), torch.tensor(_he["ref_rejected_logps"])
print(f"Held-out split (seed {HOLDOUT_SEED}): {len(trainer.train_dataset):,} training pairs, {len(_he):,} held-out, "
      f"{HOLDOUT_LEAK_DROPPED} training pairs dropped for sharing a prompt + chosen answer with a held-out pair")
for _s, _n in sorted(Counter(_he["source"]).items()):
    print(f"  held-out {_s:<24} {_n:>4}   training {Counter(trainer.train_dataset['source'])[_s]:>5}")
print(f"  held-out ref chosen mean {_hc.mean():.1f}, ref rejected mean {_hr.mean():.1f}")
# The stop rule's limit in nats: MAX_CHOSEN_DROP_PCT of the average held-out chosen log-prob.
HOLDOUT_REF_CHOSEN_MEAN = float(_hc.mean())
CHOSEN_DROP_LIMIT_NATS = abs(HOLDOUT_REF_CHOSEN_MEAN) * MAX_CHOSEN_DROP_PCT / 100.0
print(f"  stop rule: chosen answers may fall at most {MAX_CHOSEN_DROP_PCT}% = {CHOSEN_DROP_LIMIT_NATS:.2f} nats below the SFT adapter")
print(f"  {MAX_STEPS} steps x {BATCH_SIZE * GRAD_ACCUM} = {MAX_STEPS * BATCH_SIZE * GRAD_ACCUM:,} training pairs seen at most")

Loaded run 1's ref logprob cache (read-only, all 1,673 rows token-identical): /workspace/training/biblical/output/biblical_qwen3_8_27b_unsloth_4bit_dpo_run1_step25/train/ref_logprobs_cache/dataset
Ref logprob columns ready: ['ref_chosen_logps', 'ref_rejected_logps']; trainer.precompute_ref_logps = True
  ref chosen   mean -582.5 (min -1467.1, max -221.2)
  ref rejected mean -399.8 (min -545.3, max -275.4)
  SFT reference already prefers chosen on 8.3% of pairs (raw summed log-probs; longer answers score lower, so this is a sanity figure, not accuracy)
Held-out split (seed 3407): 1,547 training pairs, 120 held-out, 6 training pairs dropped for sharing a prompt + chosen answer with a held-out pair
  held-out shallow_platitude          60   training   469
  held-out voice_drift                60   training  1078
  held-out ref chosen mean -575.1, ref rejected mean -397.3
  stop rule: chosen answers may fall at most 1.0% = 5.75 nats below the SFT adapter
  100 steps x 8 = 800 training pair

## 6. Train

Run after the persistent reference-logprob cache cell. Trains for at most `MAX_STEPS`, checkpoints every `SAVE_STEPS` steps (all kept) and resumes from the newest checkpoint if one exists.

Every `EVAL_STEPS` steps the held-out pairs are scored and a `[held-out]` line is printed and appended to `TRAIN_DIR/heldout_eval.json`. **Training stops at the first evaluation where the preferred answers are more than `MAX_CHOSEN_DROP_PCT` less likely than under the SFT adapter.** Once the stop has fired, re-running this cell does not resume.

The memory callback runs every `CLEANUP_STEPS` steps, not every step, and prints `reserved`. That figure should plateau; a steady climb is the signal to stop.

In [6]:
import gc, json, os
from transformers import TrainerCallback
from transformers.trainer_utils import get_last_checkpoint


class PeriodicMemoryCleanup(TrainerCallback):
    """Collect dead Python objects, return cached CUDA blocks, and report memory every `every` steps."""

    def __init__(self, every=50):
        self.every = max(1, int(every))

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.every == 0:
            gc.collect()
            torch.cuda.empty_cache()
            print(f"  [mem] step {state.global_step}: "
                  f"allocated {torch.cuda.memory_allocated() / 2**30:.1f} GiB, "
                  f"reserved {torch.cuda.memory_reserved() / 2**30:.1f} GiB, "
                  f"peak allocated {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB")
        return control


HELDOUT_LOG = TRAIN_DIR / "heldout_eval.json"


def _write_heldout_log(payload):
    tmp_path = HELDOUT_LOG.with_suffix(".json.tmp")
    with open(tmp_path, "w") as f:
        json.dump(payload, f, indent=2)
    os.replace(tmp_path, HELDOUT_LOG)


class HeldOutRewardStop(TrainerCallback):
    """Every `every` steps, score the held-out pairs and stop once the chosen answers have dropped too far.

    Reward = beta x (policy log-prob - SFT-reference log-prob). A negative chosen reward means the
    policy is now LESS likely than the SFT adapter to produce the preferred answers: it is buying
    margin by drifting. A small drop is allowed (`limit_nats`, set from MAX_CHOSEN_DROP_PCT);
    beyond it training stops. Every evaluation is appended to HELDOUT_LOG so the Save cell can
    pick a checkpoint, also after a kernel restart.
    """

    def __init__(self, trainer, every, record, limit_nats, ref_chosen_mean):
        self.trainer, self.every, self.record = trainer, max(1, int(every)), record
        self.limit_nats, self.ref_chosen_mean = float(limit_nats), float(ref_chosen_mean)

    def on_step_end(self, args, state, control, **kwargs):
        step = state.global_step
        if step % self.every or any(e["step"] == step for e in self.record["evals"]):
            return control
        metrics = self.trainer.evaluate()   # trainer.eval_dataset; reads the cached ref log-probs
        # evaluate() leaves the model in eval mode, and DPOTrainer.log() files the training
        # metrics logged at this same step under whichever mode the model is in.
        self.trainer.model.train()
        missing = [k for k in ("eval_rewards/chosen", "eval_rewards/rejected", "eval_rewards/margins",
                               "eval_rewards/accuracies", "eval_loss") if k not in metrics]
        if missing:
            raise RuntimeError(f"Held-out evaluation returned no {missing}; got {sorted(metrics)}. "
                               "The stop rule cannot run - do not continue training blind.")
        chosen = float(metrics["eval_rewards/chosen"])
        entry = {
            "step": step,
            "rewards_chosen": chosen,
            "rewards_rejected": float(metrics["eval_rewards/rejected"]),
            "rewards_margins": float(metrics["eval_rewards/margins"]),
            "rewards_accuracies": float(metrics["eval_rewards/accuracies"]),
            "loss": float(metrics["eval_loss"]),
            "chosen_nats_vs_sft": chosen / self.trainer.beta,
        }
        # Positive = the chosen answers became LESS likely than under SFT, as % of their reference log-prob.
        entry["chosen_drop_pct"] = -entry["chosen_nats_vs_sft"] / abs(self.ref_chosen_mean) * 100.0
        entry["within_limit"] = entry["chosen_nats_vs_sft"] >= -self.limit_nats
        self.record["evals"].append(entry)
        print(f"  [held-out] step {step}: chosen reward {chosen:+.4f} ({entry['chosen_nats_vs_sft']:+.1f} nats vs SFT, "
              f"{entry['chosen_drop_pct']:+.2f}% drop, limit {MAX_CHOSEN_DROP_PCT}%) | "
              f"rejected {entry['rewards_rejected']:+.3f} | margin {entry['rewards_margins']:.3f} | "
              f"accuracy {entry['rewards_accuracies']:.3f} | loss {entry['loss']:.4f}")
        if not entry["within_limit"]:
            self.record["stopped_at"] = step
            control.should_training_stop = True
            print(f"  [held-out] chosen answers are more than {MAX_CHOSEN_DROP_PCT}% below SFT at step {step}: stopping. "
                  "The Save cell keeps the last evaluated step inside the limit.")
        _write_heldout_log(self.record)
        return control


required_ref_cols = {"ref_chosen_logps", "ref_rejected_logps"}
if not required_ref_cols.issubset(trainer.train_dataset.column_names) or not trainer.precompute_ref_logps:
    raise RuntimeError("Run the persistent reference-logprob cache cell before training.")
if trainer.eval_dataset is None or not required_ref_cols.issubset(trainer.eval_dataset.column_names):
    raise RuntimeError("No held-out set with reference log-probs. Run the cache cell (it makes the split) before training.")

last_checkpoint = get_last_checkpoint(trainer.args.output_dir)

# Held-out record: new for a new run; on a resume, keep only evaluations up to the checkpoint.
_resume_step = int(last_checkpoint.rsplit("-", 1)[-1]) if last_checkpoint else 0
heldout_record = {
    "beta": DPO_BETA, "eval_steps": EVAL_STEPS, "holdout_seed": HOLDOUT_SEED,
    "holdout_pairs": len(trainer.eval_dataset), "train_pairs": len(trainer.train_dataset),
    "leak_dropped": HOLDOUT_LEAK_DROPPED, "evals": [], "stopped_at": None,
    "learning_rate": LEARNING_RATE, "max_chosen_drop_pct": MAX_CHOSEN_DROP_PCT,
    "chosen_drop_limit_nats": CHOSEN_DROP_LIMIT_NATS, "holdout_ref_chosen_mean": HOLDOUT_REF_CHOSEN_MEAN,
}
if last_checkpoint and HELDOUT_LOG.exists():
    _previous = json.load(open(HELDOUT_LOG))
    heldout_record["evals"] = [e for e in _previous.get("evals", []) if e["step"] <= _resume_step]
    if _previous.get("stopped_at") is not None and _previous["stopped_at"] <= _resume_step:
        heldout_record["stopped_at"] = _previous["stopped_at"]
_write_heldout_log(heldout_record)

trainer.callback_handler.callbacks = [
    cb for cb in trainer.callback_handler.callbacks
    if type(cb).__name__ not in ("PeriodicMemoryCleanup", "HeldOutRewardStop", "CudaCacheClearCallback")   # stale copies from a re-run of this cell
]
trainer.add_callback(PeriodicMemoryCleanup(every=CLEANUP_STEPS))
trainer.add_callback(HeldOutRewardStop(trainer, EVAL_STEPS, heldout_record, CHOSEN_DROP_LIMIT_NATS, HOLDOUT_REF_CHOSEN_MEAN))

if heldout_record["stopped_at"] is not None:
    # The stop rule already fired. Resuming would train past the point it stopped at.
    print(f"Held-out stop already fired at step {heldout_record['stopped_at']} ({HELDOUT_LOG}). "
          "Not resuming; go to the Save cell.")
else:
    print("DPO training started...")
    print("Loss starts near 0.69 (ln 2) and should fall; rewards/margins should widen (chosen > rejected).")
    print(f"Held-out pairs are scored every {EVAL_STEPS} steps; training stops when the chosen answers fall more than "
          f"{MAX_CHOSEN_DROP_PCT}% ({CHOSEN_DROP_LIMIT_NATS:.2f} nats) below the SFT adapter.\n")
    if last_checkpoint:
        print(f"Resuming from checkpoint: {last_checkpoint}")
        result = trainer.train(resume_from_checkpoint=last_checkpoint)
    else:
        result = trainer.train()

    print("\nDPO training complete")
    print(f"Final loss: {result.training_loss:.4f}")
    print(f"Total steps: {result.global_step} of {MAX_STEPS}"
          + (" (stopped early by the held-out rule)" if heldout_record["stopped_at"] is not None else ""))

print("\nHeld-out evaluations:")
for _e in heldout_record["evals"]:
    print(f"  step {_e['step']:>4}: chosen {_e['rewards_chosen']:+.4f} ({_e['chosen_nats_vs_sft']:+.1f} nats, {_e.get('chosen_drop_pct', float('nan')):+.2f}% drop) | "
          f"rejected {_e['rewards_rejected']:+.3f} | margin {_e['rewards_margins']:.3f} | accuracy {_e['rewards_accuracies']:.3f}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


DPO training started...
Loss starts near 0.69 (ln 2) and should fall; rewards/margins should widen (chosen > rejected).
Held-out pairs are scored every 5 steps; training stops when the chosen answers fall more than 1.0% (5.75 nats) below the SFT adapter.



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,547 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 159,383,552 of 27,675,495,664 (0.58% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
5,No log,0.650630,1.701810,59988.000000,-1.329772,-1.326336,0.571461,0.011023,-0.080785,0.750000,0.091808,-575.053266,-397.620419
10,No log,0.537663,1.706483,119021.000000,-1.330038,-1.326455,0.571392,0.005847,-0.340338,0.991667,0.346185,-575.073969,-398.658632
15,No log,0.399567,1.712579,178455.000000,-1.330897,-1.326835,0.571556,0.021926,-0.708154,1.000000,0.730080,-575.009655,-400.129896
20,No log,0.309681,1.718799,237481.000000,-1.331251,-1.326936,0.571498,-0.026351,-1.073095,1.000000,1.046745,-575.202761,-401.589661
25,No log,0.236964,1.724847,296898.000000,-1.331943,-1.327313,0.571391,-0.039740,-1.413688,1.000000,1.373948,-575.256318,-402.952032
30,No log,0.185079,1.730623,356501.000000,-1.332160,-1.327333,0.570912,-0.066835,-1.742549,1.000000,1.675714,-575.364700,-404.267478
35,No log,0.145353,1.736212,415679.000000,-1.332186,-1.326947,0.571436,-0.097209,-2.067115,1.000000,1.969906,-575.486193,-405.565740
40,No log,0.119778,1.741422,475229.000000,-1.332545,-1.327216,0.571867,-0.143230,-2.352363,1.000000,2.209133,-575.670278,-406.706731
45,No log,0.098574,1.746094,534616.000000,-1.332764,-1.327239,0.571364,-0.167903,-2.613740,1.000000,2.445837,-575.768971,-407.752239
50,No log,0.083910,1.750001,594779.000000,-1.333138,-1.327370,0.571652,-0.189803,-2.830342,1.000000,2.640539,-575.856572,-408.618648


  [held-out] step 5: chosen reward +0.0110 (+0.0 nats vs SFT, -0.01% drop, limit 1.0%) | rejected -0.081 | margin 0.092 | accuracy 0.750 | loss 0.6506


... (36 outputs left out)

Unsloth: Restored added_tokens_decoder metadata in /workspace/training/biblical/output/biblical_qwen3_8_27b_unsloth_4bit_dpo_v2/train/checkpoint-95/tokenizer_config.json.


  [mem] step 100: allocated 22.7 GiB, reserved 22.9 GiB, peak allocated 24.5 GiB
  [held-out] step 100: chosen reward -0.3062 (-1.2 nats vs SFT, +0.21% drop, limit 1.0%) | rejected -3.633 | margin 3.327 | accuracy 1.000 | loss 0.0478


Unsloth: Restored added_tokens_decoder metadata in /workspace/training/biblical/output/biblical_qwen3_8_27b_unsloth_4bit_dpo_v2/train/checkpoint-100/tokenizer_config.json.



DPO training complete
Final loss: 0.1986
Total steps: 100 of 100

Held-out evaluations:
  step    5: chosen +0.0110 (+0.0 nats, -0.01% drop) | rejected -0.081 | margin 0.092 | accuracy 0.750
  step   10: chosen +0.0058 (+0.0 nats, -0.00% drop) | rejected -0.340 | margin 0.346 | accuracy 0.992
  step   15: chosen +0.0219 (+0.1 nats, -0.02% drop) | rejected -0.708 | margin 0.730 | accuracy 1.000
  step   20: chosen -0.0264 (-0.1 nats, +0.02% drop) | rejected -1.073 | margin 1.047 | accuracy 1.000
  step   25: chosen -0.0397 (-0.2 nats, +0.03% drop) | rejected -1.414 | margin 1.374 | accuracy 1.000
  step   30: chosen -0.0668 (-0.3 nats, +0.05% drop) | rejected -1.743 | margin 1.676 | accuracy 1.000
  step   35: chosen -0.0972 (-0.4 nats, +0.07% drop) | rejected -2.067 | margin 1.970 | accuracy 1.000
  step   40: chosen -0.1432 (-0.6 nats, +0.10% drop) | rejected -2.352 | margin 2.209 | accuracy 1.000
  step   45: chosen -0.1679 (-0.7 nats, +0.12% drop) | rejected -2.614 | margin 2.446 |

## 7. Save DPO LoRA Adapter

Loads the **selected checkpoint** into the `default` adapter first: the last evaluated step still inside the allowed drop (`MAX_CHOSEN_DROP_PCT`), or `SAVE_STEP_OVERRIDE` if set. If no evaluated step qualifies the cell raises and saves nothing. The Quick Evaluation cell below then generates with that same checkpoint.

Saves only the `default` adapter (not TRL's frozen `ref` copy), the tokenizer/processor files, the persona prompts from the SFT adapter, and a `complete.json` sentinel that records the selected step and every held-out evaluation.

In [7]:
import json, shutil, time, struct
from safetensors.torch import load_file
from peft.utils import set_peft_model_state_dict
from transformers.trainer_utils import get_last_checkpoint

# ---- Pick the checkpoint to save ------------------------------------------------------
# Not "whatever is in memory": training may have stopped one evaluation AFTER the last good
# step. Default = the last evaluated step still inside the allowed drop (MAX_CHOSEN_DROP_PCT).
heldout_record = json.load(open(HELDOUT_LOG))
_good_steps = [e["step"] for e in heldout_record["evals"] if e.get("within_limit", e["rewards_chosen"] >= 0)]
if SAVE_STEP_OVERRIDE is not None:
    SELECTED_STEP, _selection = int(SAVE_STEP_OVERRIDE), "SAVE_STEP_OVERRIDE"
elif _good_steps:
    SELECTED_STEP, _selection = max(_good_steps), f"last evaluated step with the chosen answers no more than {MAX_CHOSEN_DROP_PCT}% below SFT"
else:
    raise RuntimeError(
        f"No evaluated step is inside the allowed drop of {MAX_CHOSEN_DROP_PCT}% "
        f"(evaluations: {[(e['step'], round(e['rewards_chosen'], 4)) for e in heldout_record['evals']]}). "
        "Nothing was saved. By this notebook's rule the SFT adapter is the better one on these pairs. "
        "To save a checkpoint anyway for a comparison, set SAVE_STEP_OVERRIDE in Setup "
        "(or assign it here) and re-run this cell."
    )
_selected_ckpt = TRAIN_DIR / f"checkpoint-{SELECTED_STEP}"
_selected_weights = _selected_ckpt / "adapter_model.safetensors"
if not _selected_weights.exists():
    raise FileNotFoundError(f"{_selected_weights} does not exist; checkpoints present: "
                            f"{sorted(p.name for p in TRAIN_DIR.glob('checkpoint-*'))}")

# Load that checkpoint into the trainable "default" adapter, then PROVE it landed: every
# tensor in the file must equal the in-memory tensor. The frozen "ref" adapter is not touched.
_ckpt_state = load_file(str(_selected_weights))
set_peft_model_state_dict(model, _ckpt_state, adapter_name="default")
_default_params = {n: p for n, p in model.named_parameters() if ".default." in n}
_mismatched = []
for _k, _v in _ckpt_state.items():
    _head, _, _tail = _k.rpartition(".")
    _p = _default_params.get(f"{_head}.default.{_tail}")
    if _p is None or not torch.equal(_p.detach().to("cpu", torch.float32), _v.to(torch.float32)):
        _mismatched.append(_k)
if _mismatched or len(_ckpt_state) != len(_default_params):
    raise RuntimeError(
        f"checkpoint-{SELECTED_STEP} did not load cleanly into the 'default' adapter: {len(_mismatched)} of "
        f"{len(_ckpt_state)} tensors differ or are missing (first: {_mismatched[:3]}); the model has "
        f"{len(_default_params)} 'default' tensors. Nothing was saved."
    )
_selected_eval = next((e for e in heldout_record["evals"] if e["step"] == SELECTED_STEP), None)
print(f"Selected checkpoint-{SELECTED_STEP} ({_selection}); {len(_ckpt_state)} tensors loaded and verified.")

LORA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# selected_adapters: the model also carries DPOTrainer's "ref" adapter, which would
# otherwise be written to a ref/ subfolder beside the real one.
model.save_pretrained(str(LORA_OUTPUT_DIR), selected_adapters=["default"])
(processor or tokenizer).save_pretrained(str(LORA_OUTPUT_DIR))
if (LORA_OUTPUT_DIR / "ref").exists():
    shutil.rmtree(LORA_OUTPUT_DIR / "ref")
if not (LORA_OUTPUT_DIR / "adapter_model.safetensors").exists():
    raise RuntimeError(f"adapter_model.safetensors was not written to {LORA_OUTPUT_DIR}")

# Confirm what landed on disk: tensor count, dtype, no "ref" keys, same base as the SFT adapter.
with open(LORA_OUTPUT_DIR / "adapter_model.safetensors", "rb") as f:
    _header = json.loads(f.read(struct.unpack("<Q", f.read(8))[0]))
_header.pop("__metadata__", None)
_dtypes = sorted({v["dtype"] for v in _header.values()})
if any(".ref." in k for k in _header):
    raise RuntimeError("The saved adapter contains 'ref' tensors. Do not serve it.")
_saved_cfg = json.load(open(LORA_OUTPUT_DIR / "adapter_config.json"))
if _saved_cfg.get("base_model_name_or_path") != SFT_ADAPTER_BASE:
    raise RuntimeError(f"Saved adapter base is {_saved_cfg.get('base_model_name_or_path')!r}, "
                       f"the SFT adapter records {SFT_ADAPTER_BASE!r}")

persona_prompts = SFT_LORA_PATH / "persona_system_prompts.json"
if persona_prompts.exists():
    shutil.copy2(persona_prompts, LORA_OUTPUT_DIR / "persona_system_prompts.json")

metadata = {
    "status": "complete",
    "stage": "dpo",
    "model_name_base": MODEL_NAME_BASE,
    "base_model": BASE_LLM,
    "adapter_base_model": SFT_ADAPTER_BASE,
    "sft_lora_path": str(SFT_LORA_PATH),
    "dpo_data_file": str(DPO_DATA_FILE),
    "train_dir": str(TRAIN_DIR),
    "last_checkpoint": get_last_checkpoint(str(TRAIN_DIR)),
    "selected_step": SELECTED_STEP,
    "selected_checkpoint": str(_selected_ckpt),
    "selection_rule": _selection,
    "selected_heldout_eval": _selected_eval,
    "heldout_stopped_at": heldout_record.get("stopped_at"),
    "heldout_evals": heldout_record["evals"],
    "holdout_pairs": len(trainer.eval_dataset),
    "holdout_seed": HOLDOUT_SEED,
    "holdout_leak_dropped": HOLDOUT_LEAK_DROPPED,
    "global_step": getattr(trainer.state, "global_step", None),
    "max_steps": getattr(trainer.state, "max_steps", None),
    "final_loss": float(result.training_loss) if "result" in globals() else None,
    "train_pairs": len(trainer.train_dataset),
    "dpo_beta": DPO_BETA,
    "loss_type": LOSS_TYPE,
    "learning_rate": LEARNING_RATE,
    "max_chosen_drop_pct": MAX_CHOSEN_DROP_PCT,
    "chosen_drop_limit_nats": CHOSEN_DROP_LIMIT_NATS,
    "ref_logprob_cache_source": _ref_cache_source,
    "max_seq_length": MAX_SEQ_LENGTH,
    "dpo_max_pairs": DPO_MAX_PAIRS,
    "save_steps": SAVE_STEPS,
    "eval_steps": EVAL_STEPS,
    "warmup_steps": WARMUP_STEPS,
    "enable_thinking": ENABLE_THINKING,
    "reference": _ref_cache_config["reference"],
    "trl_version": trl.__version__,
    "adapter_tensors": len(_header),
    "adapter_dtypes": _dtypes,
    "formatted_dataset_fingerprint": format_fingerprint,
    "ref_logprob_cache_fingerprint": _ref_cache_fingerprint,
    "saved_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
with open(LORA_OUTPUT_DIR / "dpo_training_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
with open(LORA_OUTPUT_DIR / "complete.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved DPO LoRA to {LORA_OUTPUT_DIR}")
print(f"  {len(_header)} tensors, dtypes {_dtypes}, base {SFT_ADAPTER_BASE}")
print(f"  Sentinel: {LORA_OUTPUT_DIR / 'complete.json'} (saved step {SELECTED_STEP}; training ran to step {metadata['global_step']})")
if _selected_eval:
    print(f"  Held-out at step {SELECTED_STEP}: chosen reward {_selected_eval['rewards_chosen']:+.4f} "
          f"({_selected_eval['chosen_nats_vs_sft']:+.1f} nats vs SFT), margin {_selected_eval['rewards_margins']:.3f}, "
          f"accuracy {_selected_eval['rewards_accuracies']:.3f}")

Selected checkpoint-100 (last evaluated step with the chosen answers no more than 1.0% below SFT); 512 tensors loaded and verified.


Unsloth: Restored added_tokens_decoder metadata in /workspace/training/biblical/output/biblical_qwen3_8_27b_unsloth_4bit_dpo_v2/lora_adapters/tokenizer_config.json.


Saved DPO LoRA to /workspace/training/biblical/output/biblical_qwen3_8_27b_unsloth_4bit_dpo_v2/lora_adapters
  512 tensors, dtypes ['F32'], base unsloth/Qwen3.8-27B-unsloth-bnb-4bit
  Sentinel: /workspace/training/biblical/output/biblical_qwen3_8_27b_unsloth_4bit_dpo_v2/lora_adapters/complete.json (saved step 100; training ran to step 100)
  Held-out at step 100: chosen reward -0.3062 (-1.2 nats vs SFT), margin 3.327, accuracy 1.000


## 8. Quick Evaluation

Generates with the checkpoint the Save cell selected, using the full persona system prompts saved by the SFT run, and `ENABLE_THINKING` from the Setup cell. With it `True`, output opens with a `<think>` block, so `max_new_tokens` is raised to leave room for the reasoning plus the answer.

In [8]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)
model.set_adapter("default")   # generate with the trained adapter, not the frozen "ref" copy

_prompts_file = LORA_OUTPUT_DIR / "persona_system_prompts.json"
_persona_prompts = json.load(open(_prompts_file)) if _prompts_file.exists() else {}
eval_prompts = [
    ("voice discipline", "isaiah",
     "You are Isaiah, a court prophet who beheld God's throne and heard the seraphim cry 'Holy, holy, holy.' Speak in first person from your lived prophetic witness.",
     "What should I understand about forgiveness when I feel unworthy?"),
    ("voice discipline", "paul",
     "You are Paul, apostle to the Gentiles. Speak from your letters and lived witness.",
     "How do I keep going when the people I serve turn against me?"),
    ("depth over platitude", "job",
     "You are Job, speaking from the ash heap and wrestling honestly before God.",
     "Why does God allow righteous people to suffer?"),
]
for name, persona_key, fallback_system, user_prompt in eval_prompts:
    system_prompt = _persona_prompts.get(persona_key, fallback_system)
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=ENABLE_THINKING)
    inputs = tokenizer(text=text, return_tensors="pt").to(model.device)
    print(f"\n=== {name.upper()} / {persona_key} ({'trained persona prompt' if persona_key in _persona_prompts else 'fallback prompt'}) ===")
    print(f"USER: {user_prompt}")
    print("MODEL: ", end="")
    _ = model.generate(**inputs, max_new_tokens=1536 if ENABLE_THINKING else GEN_MAX_NEW_TOKENS, temperature=GEN_TEMPERATURE, top_p=GEN_TOP_P, top_k=GEN_TOP_K, do_sample=True, streamer=TextStreamer(tokenizer, skip_prompt=True))
del inputs


=== VOICE DISCIPLINE / isaiah (trained persona prompt) ===
USER: What should I understand about forgiveness when I feel unworthy?
MODEL: Woe to the heart that dares measure its worth by the dust of its own failures, while the throne of the Almighty trembles with the cry of seraphim—*Holy, holy, holy*—and yet, even there, in that consuming fire, a coal is taken not to burn the sinner, but to purify his lips!

Do you think the hand that stretched over Judah in wrath is too mighty to reach down in mercy? The same mouth that declared *I will cut off the head of Assyria*—that same voice now whispers over Zion: *Behold, I will heal her wounds.* The rod was lifted; it was not withdrawn in vain. It fell upon the proud, upon the rebellious, upon those who trusted in chariots and alliances forged in smoke—but it did not fall upon the contrite.

You say you are unworthy? So did I—*a man of unclean lips!*—and yet a burning coal touched my tongue. So shall it be with you: not because you have earn